In [ ]:
%pip install sklearn

In [ ]:
# Import des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgri')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("ok")

In [ ]:

print(f"Données chargées : {df.shape[0]} lignes et {df.shape[1]} colonnes")
print("\n" + "="*80)
print("Aperçu des premières lignes :")
print("="*80)
df.head()

In [ ]:
# Informations sur les colonnes et les types de données
print("Informations sur le dataset :")
print("="*80)
df.info()
print("\n" + "="*80)
print("Statistiques descriptives :")
print("="*80)
df.describe()

In [ ]:
# Vérification des valeurs manquantes
print("Valeurs manquantes par colonne :")
print("="*80)
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Nombre': missing_values,
    'Pourcentage': missing_percent.round(2)
})
print(missing_df[missing_df['Nombre'] > 0].sort_values('Nombre', ascending=False))

# Nettoyage : suppression des lignes sans customer_id
df_clean = df[df['customer_id'].notna()].copy()
print(f"\n✓ Lignes avec customer_id : {len(df_clean)} / {len(df)} ({len(df_clean)/len(df)*100:.1f}%)")

In [ ]:
# Conversion de la colonne date en format datetime
df_clean['invoice_date'] = pd.to_datetime(df_clean['invoice_date'])

print("Période des données :")
print(f"  - Date minimale : {df_clean['invoice_date'].min()}")
print(f"  - Date maximale : {df_clean['invoice_date'].max()}")
print(f"  - Durée : {(df_clean['invoice_date'].max() - df_clean['invoice_date'].min()).days} jours")

# Nombre de clients uniques
print(f"\nNombre de clients uniques : {df_clean['customer_id'].nunique()}")

## 2. Calcul des Métriques RFM

Les métriques RFM permettent d'évaluer la valeur de chaque client :
- **Récence (R)** : Nombre de jours depuis le dernier achat
- **Fréquence (F)** : Nombre total de transactions
- **Montant (M)** : Montant total dépensé

In [ ]:
# Définition de la date de référence (dernier jour du dataset + 1)
date_reference = df_clean['invoice_date'].max() + pd.Timedelta(days=1)
print(f"Date de référence pour le calcul de la Récence : {date_reference}")

# Calcul des métriques RFM par client
rfm = df_clean.groupby('customer_id').agg({
    'invoice_date': lambda x: (date_reference - x.max()).days,  # Récence
    'invoice_id': 'nunique',  # Fréquence (nombre de transactions uniques)
    'total_price': 'sum'  # Montant total
}).reset_index()

# Renommer les colonnes
rfm.columns = ['customer_id', 'Recence', 'Frequence', 'Montant']

print(f"\n✓ Métriques RFM calculées pour {len(rfm)} clients")
print("\nAperçu du DataFrame RFM :")
print("="*80)
rfm.head(10)

In [ ]:
# Statistiques descriptives des métriques RFM
print("Statistiques descriptives des métriques RFM :")
print("="*80)
rfm[['Recence', 'Frequence', 'Montant']].describe()

## 3. Analyse et Visualisation RFM

In [ ]:
# Histogrammes des distributions RFM
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(rfm['Recence'], bins=50, color='skyblue', edgecolor='black')
axes[0].set_title('Distribution de la Récence', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Jours depuis le dernier achat')
axes[0].set_ylabel('Nombre de clients')
axes[0].grid(alpha=0.3)

axes[1].hist(rfm['Frequence'], bins=50, color='lightcoral', edgecolor='black')
axes[1].set_title('Distribution de la Fréquence', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Nombre de transactions')
axes[1].set_ylabel('Nombre de clients')
axes[1].grid(alpha=0.3)

axes[2].hist(rfm['Montant'], bins=50, color='lightgreen', edgecolor='black')
axes[2].set_title('Distribution du Montant', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Montant total dépensé (€)')
axes[2].set_ylabel('Nombre de clients')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots pour identifier les valeurs aberrantes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].boxplot(rfm['Recence'], vert=True)
axes[0].set_title('Boxplot de la Récence', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jours')
axes[0].grid(alpha=0.3)

axes[1].boxplot(rfm['Frequence'], vert=True)
axes[1].set_title('Boxplot de la Fréquence', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Nombre de transactions')
axes[1].grid(alpha=0.3)

axes[2].boxplot(rfm['Montant'], vert=True)
axes[2].set_title('Boxplot du Montant', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Montant (€)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation entre les métriques RFM
correlation_matrix = rfm[['Recence', 'Frequence', 'Montant']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation des Métriques RFM', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nAnalyse de corrélation :")
print("="*80)
print(correlation_matrix)

## 4. Segmentation RFM Manuelle

Création de scores RFM basés sur des quintiles (1-5) pour chaque métrique, puis segmentation en catégories de clients.

In [ ]:
# Calcul des scores RFM en quintiles (1-5)
# Note : Pour Récence, un score élevé = faible récence (meilleur)
rfm['R_Score'] = pd.qcut(rfm['Recence'], q=5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequence'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Montant'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# Conversion en entiers
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

# Score RFM combiné (simple concaténation)
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# Score RFM agrégé (moyenne)
rfm['RFM_Score_Total'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print("Scores RFM calculés ✓")
print("\nAperçu des scores :")
rfm[['customer_id', 'Recence', 'Frequence', 'Montant', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'RFM_Score_Total']].head(10)

In [ ]:
# Fonction de segmentation RFM
def segment_rfm(row):
    """Assigne un segment client basé sur les scores RFM"""
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    
    # Champions : Meilleurs clients (R=5, F=5, M=5 ou proche)
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    
    # Clients Fidèles : Achats fréquents mais montants variables
    elif r >= 3 and f >= 4:
        return 'Clients Fidèles'
    
    # Clients Potentiels : Récence élevée, faible fréquence
    elif r >= 4 and f <= 2:
        return 'Clients Potentiels'
    
    # Clients à Risque : Récence moyenne-faible, fréquence élevée
    elif r <= 2 and f >= 3:
        return 'À Risque'
    
    # Clients Endormis : Récence faible, fréquence moyenne
    elif r <= 2 and f >= 2 and f <= 3:
        return 'Endormis'
    
    # Clients Perdus : Très faible récence
    elif r <= 2 and f <= 2:
        return 'Perdus'
    
    # Clients Occasionnels : Score moyen partout
    else:
        return 'Occasionnels'

# Application de la segmentation
rfm['Segment'] = rfm.apply(segment_rfm, axis=1)

print("Segmentation RFM réalisée ✓")
print("\nDistribution des segments :")
print("="*80)
segment_counts = rfm['Segment'].value_counts().sort_values(ascending=False)
print(segment_counts)
print(f"\nTotal : {segment_counts.sum()} clients")

In [ ]:
# Visualisation de la distribution des segments
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Diagramme en barres
segment_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Nombre de Clients par Segment RFM', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Nombre de clients')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3, axis='y')

# Diagramme circulaire
segment_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Répartition des Segments RFM', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Statistiques moyennes par segment
segment_stats = rfm.groupby('Segment')[['Recence', 'Frequence', 'Montant']].mean().round(2)
segment_stats['Nombre_Clients'] = rfm.groupby('Segment').size()
segment_stats = segment_stats.sort_values('Nombre_Clients', ascending=False)

print("Profil moyen de chaque segment :")
print("="*80)
segment_stats

## 5. Préparation pour le Clustering

Normalisation des données RFM pour l'algorithme K-Means.

In [ ]:
# Sélection des features pour le clustering
features = ['Recence', 'Frequence', 'Montant']
X = rfm[features].copy()

print("Données avant normalisation :")
print("="*80)
print(X.describe())

# Normalisation avec StandardScaler (moyenne=0, écart-type=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Création d'un DataFrame pour visualisation
X_scaled_df = pd.DataFrame(X_scaled, columns=[f'{col}_scaled' for col in features])

print("\n" + "="*80)
print("Données après normalisation (StandardScaler) :")
print("="*80)
print(X_scaled_df.describe())

print("\n✓ Normalisation effectuée avec succès")

In [ ]:
# Visualisation avant/après normalisation
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, col in enumerate(features):
    # Avant normalisation
    axes[0, i].hist(X[col], bins=50, color='orange', edgecolor='black', alpha=0.7)
    axes[0, i].set_title(f'{col} - Avant Normalisation', fontsize=12, fontweight='bold')
    axes[0, i].set_ylabel('Fréquence')
    axes[0, i].grid(alpha=0.3)
    
    # Après normalisation
    axes[1, i].hist(X_scaled[:, i], bins=50, color='green', edgecolor='black', alpha=0.7)
    axes[1, i].set_title(f'{col} - Après Normalisation', fontsize=12, fontweight='bold')
    axes[1, i].set_ylabel('Fréquence')
    axes[1, i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Application du Clustering K-Means

Détermination du nombre optimal de clusters et application de l'algorithme K-Means.

In [ ]:
# Méthode du coude (Elbow Method)
inertias = []
silhouette_scores = []
K_range = range(2, 11)

print("Calcul de l'inertie et du score de silhouette pour différents nombres de clusters...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))
    print(f"  k={k} : Inertie={kmeans.inertia_:.2f}, Silhouette={silhouette_scores[-1]:.4f}")

print("\n✓ Calculs terminés")

In [ ]:
# Visualisation de la méthode du coude et du score de silhouette
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Méthode du coude
axes[0].plot(K_range, inertias, marker='o', linewidth=2, markersize=8, color='blue')
axes[0].set_title('Méthode du Coude (Elbow Method)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Nombre de Clusters (k)')
axes[0].set_ylabel('Inertie (Within-Cluster Sum of Squares)')
axes[0].grid(alpha=0.3)
axes[0].axvline(x=4, color='red', linestyle='--', label='k=4 (suggéré)')
axes[0].legend()

# Score de silhouette
axes[1].plot(K_range, silhouette_scores, marker='s', linewidth=2, markersize=8, color='green')
axes[1].set_title('Score de Silhouette', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Nombre de Clusters (k)')
axes[1].set_ylabel('Score de Silhouette')
axes[1].grid(alpha=0.3)

# Trouver le meilleur k selon le score de silhouette
best_k = K_range[np.argmax(silhouette_scores)]
axes[1].axvline(x=best_k, color='red', linestyle='--', label=f'k={best_k} (optimal)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nMeilleur nombre de clusters selon le score de silhouette : {best_k}")

In [ ]:
# Application du K-Means avec le nombre optimal de clusters
optimal_k = 4  # À ajuster selon les résultats ci-dessus

kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
rfm['Cluster'] = kmeans_final.fit_predict(X_scaled)

print(f"✓ K-Means appliqué avec {optimal_k} clusters")
print(f"\nInertie finale : {kmeans_final.inertia_:.2f}")
print(f"Score de silhouette final : {silhouette_score(X_scaled, rfm['Cluster']):.4f}")

print("\nDistribution des clusters :")
print("="*80)
print(rfm['Cluster'].value_counts().sort_index())

## 7. Évaluation et Visualisation des Clusters

In [ ]:
# Profil moyen de chaque cluster
cluster_profile = rfm.groupby('Cluster')[['Recence', 'Frequence', 'Montant']].mean().round(2)
cluster_profile['Nombre_Clients'] = rfm.groupby('Cluster').size()
cluster_profile['Pourcentage'] = (cluster_profile['Nombre_Clients'] / len(rfm) * 100).round(2)

print("Profil des Clusters :")
print("="*80)
cluster_profile

In [ ]:
# Visualisation 3D des clusters
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Couleurs pour chaque cluster
colors = plt.cm.viridis(np.linspace(0, 1, optimal_k))

for cluster in range(optimal_k):
    cluster_data = rfm[rfm['Cluster'] == cluster]
    ax.scatter(cluster_data['Recence'], 
               cluster_data['Frequence'], 
               cluster_data['Montant'],
               c=[colors[cluster]], 
               label=f'Cluster {cluster}',
               s=50, 
               alpha=0.6,
               edgecolors='black')

ax.set_xlabel('Récence (jours)', fontsize=12)
ax.set_ylabel('Fréquence (transactions)', fontsize=12)
ax.set_zlabel('Montant (€)', fontsize=12)
ax.set_title('Visualisation 3D des Clusters RFM', fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Réduction de dimensionnalité avec PCA pour visualisation 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Variance expliquée par les 2 premières composantes principales :")
print(f"  - PC1 : {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"  - PC2 : {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"  - Total : {sum(pca.explained_variance_ratio_)*100:.2f}%")

# Visualisation 2D avec PCA
plt.figure(figsize=(14, 8))

for cluster in range(optimal_k):
    cluster_mask = rfm['Cluster'] == cluster
    plt.scatter(X_pca[cluster_mask, 0], 
                X_pca[cluster_mask, 1], 
                c=[colors[cluster]],
                label=f'Cluster {cluster}',
                s=50, 
                alpha=0.6,
                edgecolors='black')

# Afficher les centres des clusters
centers_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], 
            c='red', marker='X', s=300, edgecolors='black', 
            linewidths=2, label='Centres')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
plt.title('Visualisation 2D des Clusters (PCA)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap des caractéristiques moyennes par cluster
cluster_means_normalized = rfm.groupby('Cluster')[['Recence', 'Frequence', 'Montant']].mean()

# Normalisation pour la heatmap (MinMaxScaler pour chaque métrique)
from sklearn.preprocessing import MinMaxScaler
scaler_heatmap = MinMaxScaler()
cluster_means_normalized_values = scaler_heatmap.fit_transform(cluster_means_normalized)
cluster_means_normalized = pd.DataFrame(
    cluster_means_normalized_values,
    columns=cluster_means_normalized.columns,
    index=cluster_means_normalized.index
)

plt.figure(figsize=(10, 6))
sns.heatmap(cluster_means_normalized.T, annot=True, fmt='.2f', cmap='RdYlGn_r', 
            linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Profil Normalisé des Clusters (0=Min, 1=Max)', fontsize=14, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Métrique RFM')
plt.tight_layout()
plt.show()

## 8. Interprétation et Recommandations

In [ ]:
# Fonction pour nommer et interpréter les clusters
def interpreter_cluster(cluster_id, profile):
    """Interprète un cluster basé sur son profil RFM"""
    r = profile['Recence']
    f = profile['Frequence']
    m = profile['Montant']
    
    interpretations = {
        'nom': f'Cluster {cluster_id}',
        'description': '',
        'strategie': ''
    }
    
    # Logique d'interprétation basée sur les moyennes
    if r < 100 and f > 100 and m > 2000:
        interpretations['nom'] = 'Champions'
        interpretations['description'] = 'Clients très actifs, achats récents, fréquents et montants élevés'
        interpretations['strategie'] = 'Programme VIP, offres exclusives, récompenses de fidélité'
    
    elif r < 150 and f > 50:
        interpretations['nom'] = 'Clients Fidèles'
        interpretations['description'] = 'Clients réguliers avec bonne récence et fréquence'
        interpretations['strategie'] = 'Cross-selling, upselling, recommandations personnalisées'
    
    elif r > 200 and m > 1000:
        interpretations['nom'] = 'Clients à Réactiver'
        interpretations['description'] = 'Clients de valeur mais inactifs depuis longtemps'
        interpretations['strategie'] = 'Campagnes de réactivation, offres de retour, enquêtes de satisfaction'
    
    elif f < 30 and m < 500:
        interpretations['nom'] = 'Nouveaux/Occasionnels'
        interpretations['description'] = 'Clients peu actifs ou nouveaux'
        interpretations['strategie'] = 'Onboarding, incentives pour premier achat, programmes de bienvenue'
    
    else:
        interpretations['nom'] = f'Segment {cluster_id}'
        interpretations['description'] = 'Profil mixte nécessitant une analyse approfondie'
        interpretations['strategie'] = 'Segmentation supplémentaire recommandée'
    
    return interpretations

# Application de l'interprétation
interpretations = []
for cluster_id in range(optimal_k):
    profile = cluster_profile.loc[cluster_id]
    interp = interpreter_cluster(cluster_id, profile)
    interp['cluster_id'] = cluster_id
    interp['nb_clients'] = profile['Nombre_Clients']
    interp['pourcentage'] = profile['Pourcentage']
    interp['recence_moy'] = profile['Recence']
    interp['frequence_moy'] = profile['Frequence']
    interp['montant_moy'] = profile['Montant']
    interpretations.append(interp)

interp_df = pd.DataFrame(interpretations)

print("Interprétation des Clusters :")
print("="*80)
for _, row in interp_df.iterrows():
    print(f"\n🔹 {row['nom']} (Cluster {row['cluster_id']})")
    print(f"   Clients : {row['nb_clients']} ({row['pourcentage']}%)")
    print(f"   Profil : R={row['recence_moy']:.0f}j, F={row['frequence_moy']:.0f}, M={row['montant_moy']:.0f}€")
    print(f"   Description : {row['description']}")
    print(f"   Stratégie : {row['strategie']}")

In [ ]:
# Tableau récapitulatif final
summary_df = interp_df[['cluster_id', 'nom', 'nb_clients', 'pourcentage', 
                         'recence_moy', 'frequence_moy', 'montant_moy']].copy()
summary_df.columns = ['Cluster', 'Nom', 'Nb Clients', '%', 'Récence Moy', 'Fréquence Moy', 'Montant Moy']
summary_df = summary_df.sort_values('Nb Clients', ascending=False)

print("\n" + "="*80)
print("TABLEAU RÉCAPITULATIF DES SEGMENTS")
print("="*80)
summary_df

In [ ]:
# Export des résultats
# Sauvegarde du DataFrame RFM avec les clusters
rfm_export = rfm[['customer_id', 'Recence', 'Frequence', 'Montant', 
                   'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 
                   'Segment', 'Cluster']].copy()

rfm_export.to_csv('../data/rfm_clusters_results.csv', index=False)
print("✓ Résultats exportés vers : ../data/rfm_clusters_results.csv")

# Sauvegarde du profil des clusters
cluster_profile.to_csv('../data/cluster_profiles.csv')
print("✓ Profils des clusters exportés vers : ../data/cluster_profiles.csv")

---

## Conclusion

Cette analyse a permis de :
1. ✅ Calculer les métriques RFM pour chaque client
2. ✅ Identifier des segments clients via une segmentation RFM manuelle
3. ✅ Appliquer un clustering K-Means pour découvrir des groupes homogènes
4. ✅ Visualiser et interpréter les résultats

### Recommandations Générales

**Actions prioritaires :**
- Mettre en place des campagnes personnalisées pour chaque segment
- Surveiller les clients à risque et mettre en place des actions de rétention
- Récompenser les Champions avec des programmes de fidélité exclusifs
- Réactiver les clients endormis avec des offres ciblées

**Prochaines étapes :**
- Intégrer ces segments dans le CRM
- Automatiser le recalcul des scores RFM (mensuel/trimestriel)
- A/B tester les stratégies marketing par segment
- Analyser l'évolution des clients entre segments au fil du temps